In [26]:
import pandas as pd
import numpy as np

from tqdm import tqdm

books = pd.read_csv("books_with_categories.csv")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None, device="cuda")

classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528673090040684},
  {'label': 'neutral', 'score': 0.005764594301581383},
  {'label': 'anger', 'score': 0.004419779404997826},
  {'label': 'sadness', 'score': 0.002092393347993493},
  {'label': 'disgust', 'score': 0.001611992483958602},
  {'label': 'fear', 'score': 0.00041385195800103247}]]

In [7]:
# testing on an actual book!
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [8]:
classifier(books["description"][0])

[[{'label': 'fear', 'score': 0.6548418402671814},
  {'label': 'neutral', 'score': 0.16985197365283966},
  {'label': 'sadness', 'score': 0.11640845984220505},
  {'label': 'surprise', 'score': 0.020700769498944283},
  {'label': 'disgust', 'score': 0.01910068280994892},
  {'label': 'joy', 'score': 0.015161238610744476},
  {'label': 'anger', 'score': 0.0039351521991193295}]]

In [9]:
classifier(books["description"][0].split(".")) # sentiment analysis on indiv sentences

[[{'label': 'surprise', 'score': 0.729603111743927},
  {'label': 'neutral', 'score': 0.1403854340314865},
  {'label': 'fear', 'score': 0.06816213577985764},
  {'label': 'joy', 'score': 0.04794234409928322},
  {'label': 'anger', 'score': 0.009156344458460808},
  {'label': 'disgust', 'score': 0.0026284705381840467},
  {'label': 'sadness', 'score': 0.0021221584174782038}],
 [{'label': 'neutral', 'score': 0.44937101006507874},
  {'label': 'disgust', 'score': 0.27359122037887573},
  {'label': 'joy', 'score': 0.10908293724060059},
  {'label': 'sadness', 'score': 0.09362731128931046},
  {'label': 'anger', 'score': 0.04047826677560806},
  {'label': 'surprise', 'score': 0.026970185339450836},
  {'label': 'fear', 'score': 0.006879061926156282}],
 [{'label': 'neutral', 'score': 0.6462167501449585},
  {'label': 'sadness', 'score': 0.24273262917995453},
  {'label': 'disgust', 'score': 0.04342261329293251},
  {'label': 'surprise', 'score': 0.028300542384386063},
  {'label': 'joy', 'score': 0.0142114

In [12]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [14]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.009156344458460808},
 {'label': 'disgust', 'score': 0.0026284705381840467},
 {'label': 'fear', 'score': 0.06816213577985764},
 {'label': 'joy', 'score': 0.04794234409928322},
 {'label': 'neutral', 'score': 0.1403854340314865},
 {'label': 'sadness', 'score': 0.0021221584174782038},
 {'label': 'surprise', 'score': 0.729603111743927}]

In [23]:
emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [30]:
emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [10:13<00:00,  8.47it/s]   


In [32]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [33]:
emotions_df.head()

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.064134,0.273591,0.928168,0.932797,0.646217,0.967158,0.729603,9780002005883
1,0.612618,0.348285,0.942528,0.704421,0.887940,0.111690,0.252545,9780002261982
2,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078766,9780006178736
3,0.351483,0.150723,0.360707,0.251881,0.732686,0.111690,0.078766,9780006280897
4,0.081412,0.184495,0.095043,0.040564,0.884390,0.475881,0.078766,9780006280934


In [34]:
books = pd.merge(books, emotions_df, on="isbn13")

In [38]:
books.to_csv("books_with_emotions.csv", index=False)